# 📚 Smart Document Q&A with RAG
**Retrieval-Augmented Generation for Legal/Technical Documents**

_End-to-End RAG System — Self-Hosted with Chroma & Open Source Models_

---

## What I Learned in Week 5

Week 5 covered the core building blocks of RAG (Retrieval-Augmented Generation):
- **RAG fundamentals**: Combining LLM with knowledge base for accurate Q&A
- **Chroma vector database**: Storing embeddings for semantic search
- **Text embeddings**: Converting text to numerical representations
- **Document chunking**: Splitting documents for optimal retrieval
- **Semantic search**: Finding relevant docs by meaning, not keywords
- **Evaluation**: Measuring RAG system accuracy (evals)
- **LangChain 1.0**: Modern RAG pipeline construction

## Why This Project

As a developer, I need a tool that can:
1. Answer questions about my codebase/contracts/docs
2. Cite sources for every answer
3. Work with my own documents (no API training)
4. Provide accurate answers grounded in context
5. Run locally for privacy

This exercise builds a production-ready RAG system for legal/technical documents.

---

## Components Used

| Component | Purpose |
|---|---|
| `crawler` (requests + BeautifulSoup) | Same-domain crawl, 200-link limit, stats |
| `bertopic` + `transformers` (zero-shot) | BERTopic topics, zero-shot category names (configurable models) |
| `chromadb` | Persistent vector database at data/db |
| `OpenAIEmbeddings` | Convert text to vectors |
| `RecursiveCharacterTextSplitter` or LLM chunking | Document chunking (set `USE_LLM_CHUNKING`) |
| Configurable LLM clients | Chunking, reranking, and chat (model / base_url / api_key per task; OpenAI or Ollama) |
| `evaluation/` (tests.jsonl, eval) | Retrieval + answer evals (keyword coverage, LLM judge) |
| `gr.Blocks` + Tabs | Chat UI with streaming, Sources tab (extracted docs + links), Evaluation tab with charts |

---

## How It Works

1. **Crawl** → Visit website, follow same-domain links (max 200), extract content; BERTopic on all pages for content-based categories (zero-shot topic names). Re-run to re-categorize existing crawl.
2. **Store** → Save to `data/source/<domain>/<category>/` with source URL per page; optional _summary.md
3. **Load & Chunk** → Read from data/source, split into semantic sections (~500 chars each)
3. **Embed** → Convert chunks to vectors using OpenAI
4. **Store** → Save to Chroma at `data/db`
5. **Query** → User asks question, convert to embedding
6. **Retrieve** → Find similar chunks using cosine similarity
7. **Generate** → Pass context + question to LLM
8. **Respond** → Answer with source URL citations

---

## RAG Pipeline Architecture

```
[User Query]
       ↓
[Embed Query] → OpenAI Embeddings
       ↓
[Semantic Search] → Chroma (RETRIEVAL_K chunks)
       ↓
[LLM Re-rank] → Top RERANK_TOP_K chunks
       ↓
[Generate Answer] → LLM with context
       ↓
[Response + Sources] → Final output
```

**Optimizations:** Set `USE_LLM_CHUNKING = True` in config for LLM-based chunking. Use the **Evaluation** tab in the Gradio UI (or Cell 6b) to run retrieval (MRR, keyword coverage) and answer (LLM-as-judge) metrics over `evaluation/tests.jsonl`. Chat uses streaming; Sources tab shows extracted docs with links for the last response.

---

**Hardware:** Runs on CPU with optional GPU for embeddings

## Cell 1 — Install Dependencies

In [9]:
# Run once — restart kernel after installation
!uv pip install -q chromadb langchain-openai langchain-core tiktoken gradio plotly scikit-learn requests beautifulsoup4 transformers bertopic sentence-transformers umap-learn hdbscan

print("Dependencies installed. Restart kernel if needed.")

Dependencies installed. Restart kernel if needed.


## Cell 2 — Configuration & Imports

In [11]:
import os
import glob
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
import gradio as gr
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go


# Configuration
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY", "")

if openai_api_key:
    print(f"OpenAI API Key: {openai_api_key[:8]}...")


else:
    print("⚠️ No OpenAI key found!")


CLIENT_TYPE = "chat"  "chunking" "reranking"

OPENAI_MODEL = "gpt-4.1-nano"


def get_openai_client(client_type: CLIENT_TYPE):
    if not openai_api_key:
        return get_ollama_client()
    if client_type == "chat":
        return OpenAI(model="gpt-4.1-nano", api_key=openai_api_key)
    elif client_type == "chunking":
        return OpenAI(model="gpt-4.1-nano", api_key=openai_api_key)
    elif client_type == "reranking":
        return OpenAI(model="gpt-4.1-nano", api_key=openai_api_key)

def get_ollama_client():
    OLLAMA_MODEL = "llama3.2"
    OLLAMA_BASE_URL = "http://localhost:11434/v1"
    return OpenAI(base_url=OLLAMA_BASE_URL, model=OLLAMA_MODEL,api_key="ollama")

EMBEDDING_MODEL = "text-embedding-3-small"
COLLECTION_NAME = "documents"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
RETRIEVAL_K = 10
RERANK_TOP_K = 3
USE_LLM_CHUNKING = True


SENTENCE_TRANSFORMER_MODEL = "all-MiniLM-L6-v2"
ZERO_SHOT_CLASSIFIER_MODEL = "typeform/distilbert-base-uncased-mnli"
HDBSCAN_MIN_CLUSTER_SIZE = 4
HDBSCAN_MIN_SAMPLES = 2

DATA_DIR = Path("data")
DATA_SOURCE = DATA_DIR / "source"
DB_PATH = DATA_DIR / "db"
BASE_URL = "https://topfaith.sch.ng"




embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

from evaluation.eval import set_judge_client
set_judge_client(get_openai_client("chat"), OPENAI_MODEL)

print("✅ Configuration ready.", {
    "openai_model": OPENAI_MODEL,
    "embedding_model": EMBEDDING_MODEL,
    "collection_name": COLLECTION_NAME,
    
})

AttributeError: partially initialized module 'pandas' has no attribute 'core' (most likely due to a circular import)

## Cell 3 — Knowledge Base from Website Crawl

In [25]:
from urllib.parse import urlparse
import re
import shutil
import numpy as np
from crawler import crawl_site
from transformers import pipeline
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN

def get_domain(url: str) -> str:
    return urlparse(url).netloc or "unknown"

def sanitize_category(name: str) -> str:
    s = (name or "").lower().replace(" ", "_").replace("-", "_").strip()
    s = re.sub(r"[^a-z0-9_]", "", s)[:30]
    return s or "general"


def slug_from_title(title: str, url: str = "", max_len: int = 35) -> str:
    raw = (title or "").strip()
    if not raw or raw.lower() == "no title":
        if url:
            path = urlparse(url).path.strip("/")
            raw = path.split("/")[-1] if path else "page"
        else:
            raw = "page"
    s = raw.lower().replace(" ", "_").replace("-", "_")
    s = re.sub(r"[^a-z0-9_]", "", s).strip("_")[:max_len]
    return s or "page"

CATEGORY_STOPLIST = frozenset({
    "and", "the", "for", "are", "but", "not", "you", "all", "can", "had", "her", "his", "was", "one",
    "our", "out", "has", "how", "its", "may", "now", "own", "say", "see", "way", "who", "did", "get",
    "click", "read", "more", "here", "page", "form", "back", "next", "prev", "view", "link", "menu",
})
MIN_CATEGORY_LEN = 3

def is_number_or_contact_like(sanitized: str) -> bool:
    if not sanitized:
        return False
    return all(c.isdigit() for c in sanitized)

def is_meaningful_category(sanitized: str) -> bool:
    if len(sanitized) < MIN_CATEGORY_LEN or sanitized in CATEGORY_STOPLIST:
        return False
    if is_number_or_contact_like(sanitized):
        return False
    return True

def filter_topic_words(words_tuples: list, max_words: int = 5) -> list:
    if not words_tuples:
        return []
    allowed = [w for w, _ in words_tuples if len(w) >= MIN_CATEGORY_LEN and w.lower() not in CATEGORY_STOPLIST and not is_number_or_contact_like(re.sub(r"[^a-z0-9]", "", w.lower()))]
    return allowed[:max_words]

def load_pages_from_md(domain_dir: Path) -> list[dict]:
    pages = []
    for md_path in domain_dir.rglob("*.md"):
        if md_path.name == "_summary.md":
            continue
        raw = md_path.read_text(encoding="utf-8")
        url, title, text = "", "No title", raw
        if raw.startswith("Source: "):
            first_line, _, rest = raw.partition("\n\n")
            url = first_line.replace("Source: ", "").strip()
            parts = rest.split("\n\n", 1)
            if parts and parts[0].startswith("# "):
                title = parts[0].replace("# ", "").strip()
                text = parts[1] if len(parts) > 1 else ""
            else:
                text = rest
        pages.append({"url": url, "title": title, "text": text})
    return pages

def categorize_and_save(pages: list[dict], domain_dir: Path) -> None:
    docs_for_bertopic = [(p["title"] + " " + (p["text"] or "")[:500])[:512] for p in pages]
    hdbscan_model = HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=HDBSCAN_MIN_SAMPLES,
        cluster_selection_method="eom",
        prediction_data=True,
    )
    embedding_model = SentenceTransformer(SENTENCE_TRANSFORMER_MODEL)
    topic_model = BERTopic(embedding_model=embedding_model, hdbscan_model=hdbscan_model)
    topics, probs = topic_model.fit_transform(docs_for_bertopic)
    topics = np.asarray(topics)
    if probs is not None:
        probs = np.asarray(probs)
    classifier = pipeline("zero-shot-classification", model=ZERO_SHOT_CLASSIFIER_MODEL)
    topic_label_map = {-1: "general"}
    unique_topics = sorted(set(topics))
    for topic_id in unique_topics:
        if topic_id == -1:
            continue
        words_tuples = topic_model.get_topic(topic_id) or []
        words = filter_topic_words(words_tuples, max_words=5)
        if not words:
            topic_label_map[topic_id] = "general"
            continue
        indices = np.where(topics == topic_id)[0]
        if probs is not None and probs.ndim == 2 and topic_id < probs.shape[1]:
            j = int(indices[np.argmax(probs[indices, topic_id])])
        else:
            j = int(indices[0])
        rep_doc = docs_for_bertopic[j]
        out = classifier(rep_doc, words, multi_label=False)
        chosen = "general"
        for label in out["labels"]:
            cand = sanitize_category(label)
            if is_meaningful_category(cand):
                chosen = cand
                break
        if is_number_or_contact_like(chosen):
            chosen = "contacts"
        topic_label_map[topic_id] = chosen
    used_in_cat = {}
    for idx, page in enumerate(pages):
        tid = int(topics[idx])
        cat = topic_label_map.get(tid, "general")
        cat_dir = domain_dir / cat
        cat_dir.mkdir(parents=True, exist_ok=True)
        base_slug = slug_from_title(page["title"], page["url"])
        used = used_in_cat.setdefault(cat, set())
        name = base_slug
        n = 1
        while name in used:
            n += 1
            name = f"{base_slug}_{n}"
        used.add(name)
        path = cat_dir / f"{name}.md"
        content = f"Source: {page['url']}\n\n# {page['title']}\n\n{page['text']}"
        path.write_text(content, encoding="utf-8")
    n_cats = len(used_in_cat)
    print(f"Saved {len(pages)} pages under {domain_dir}: {n_cats} content-based categories")

domain = get_domain(BASE_URL)
domain_dir = DATA_SOURCE / domain
domain_dir.mkdir(parents=True, exist_ok=True)

md_files = [p for p in domain_dir.glob("**/*.md") if p.name != "_summary.md"]
if md_files:
    print(f"Re-categorizing {len(md_files)} existing pages at {domain_dir} (content-based)...")
    pages = load_pages_from_md(domain_dir)
    for subdir in domain_dir.iterdir():
        if subdir.is_dir():
            shutil.rmtree(subdir)
    if not pages:
        pages = [{"url": BASE_URL, "title": "Demo", "text": "No content to re-categorize."}]
        (domain_dir / "general").mkdir(exist_ok=True)
        (domain_dir / "general" / "demo.md").write_text(
            f"Source: {BASE_URL}\n\n# Demo\n\n{pages[0]['text']}", encoding="utf-8"
        )
        print("No pages loaded; created general/demo.md")
    else:
        categorize_and_save(pages, domain_dir)
else:
    print(f"Crawling {BASE_URL} (max 200 links, same-domain only)...")
    pages, stats = crawl_site(BASE_URL, output_dir=domain_dir, max_links=200)
    print(f"Pages visited: {stats['total_pages_visited']}, Links discovered: {stats['total_links_discovered']}")
    if not pages:
        fallback_dir = domain_dir / "general"
        fallback_dir.mkdir(exist_ok=True)
        (fallback_dir / "demo.md").write_text(
            "Source: https://example.com\n\n# Demo\n\nNo pages could be crawled. Using demo content for RAG.",
            encoding="utf-8",
        )
        print("No pages crawled; created demo.md for RAG.")
    else:
        categorize_and_save(pages, domain_dir)

if (domain_dir / "_summary.md").exists():
    print((domain_dir / "_summary.md").read_text(encoding="utf-8"))

Re-categorizing 28 existing pages at data/source/topfaith.sch.ng (content-based)...


Device set to use mps:0


Saved 28 pages under data/source/topfaith.sch.ng: 5 content-based categories
# Crawl summary

- Domain: topfaith.sch.ng
- Crawl date: 2026-03-02 23:08 UTC
- Total pages visited: 28
- Total links discovered (same-domain): 27



## Cell 4 — Document Processing & Embedding

In [26]:
from langchain_community.docstore.document import Document

documents = []
for md_path in domain_dir.rglob("*.md"):
    if md_path.name == "_summary.md":
        continue
    raw = md_path.read_text(encoding="utf-8")
    url = ""
    if raw.startswith("Source: "):
        first_line, _, body = raw.partition("\n\n")
        url = first_line.replace("Source: ", "").strip()
        raw = body
    category = md_path.parent.name
    doc = Document(
        page_content=raw,
        metadata={"source": md_path.name, "url": url, "category": category},
    )
    documents.append(doc)

print(f"📄 Loaded {len(documents)} documents from {domain_dir}")

if USE_LLM_CHUNKING:
    print("Using LLM chunking")
    import json
    chunks = []
    for i, doc in enumerate(documents):
        if (i + 1) % 5 == 0 or i == 0:
            print(f"LLM chunking doc {i + 1}/{len(documents)}...")
        meta = doc.metadata
        how_many = max(1, len(doc.page_content) // CHUNK_SIZE)
        prompt = f"""Split this document into about {how_many} overlapping chunks for a Q&A knowledge base.
For each chunk provide: headline (short phrase likely to match user queries), summary (2-3 sentences), and original_text (exact text from the document).
Cover the whole document with ~25% overlap. Return JSON only: {{"chunks": [{{"headline": "...", "summary": "...", "original_text": "..."}}, ...]}}

Document (source: {meta.get('source', '')}):
{doc.page_content[:12000]}"""
        resp = llm_client_chunking.chat.completions.create(
            model=LLM_CHUNKING_MODEL,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0,
        )
        data = json.loads(resp.choices[0].message.content)
        for c in data.get("chunks", []):
            content = f"{c.get('headline', '')}\n\n{c.get('summary', '')}\n\n{c.get('original_text', '')}"
            chunks.append(Document(page_content=content, metadata=dict(meta)))
else:
    print("Using default chunking with character splitter")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
    )
    chunks = text_splitter.split_documents(documents)
print(f"✂️ Split into {len(chunks)} chunks")

for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i} ---")
    print(f"Source: {chunk.metadata.get('source')}, URL: {chunk.metadata.get('url', '')[:50]}...")
    print(f"Content: {chunk.page_content[:200]}...")

📄 Loaded 28 documents from data/source/topfaith.sch.ng
Using LLM chunking
LLM chunking doc 1/28...
LLM chunking doc 5/28...
LLM chunking doc 10/28...
LLM chunking doc 15/28...
LLM chunking doc 20/28...
LLM chunking doc 25/28...
✂️ Split into 102 chunks

--- Chunk 0 ---
Source: test_links__topfaith_schools.md, URL: https://topfaith.sch.ng/pg/admissions/test-links...
Content: Contact Information and Test Links

Topfaith Schools provides contact details including phone numbers and email for inquiries. The document also includes links for various admission tests with specifi...

--- Chunk 1 ---
Source: test_links__topfaith_schools.md, URL: https://topfaith.sch.ng/pg/admissions/test-links...
Content: Admission Test Links and Access Codes

Candidates can access admission tests for JS1, JS2, JS3, SS1, and SS2 levels through provided links, each secured with unique access codes. These links facilitat...

--- Chunk 2 ---
Source: test_links__topfaith_schools.md, URL: https://topfaith.sch.ng/pg/a

## Cell 5 — Chroma Vector Store

In [27]:
from chromadb import PersistentClient

DB_PATH.mkdir(parents=True, exist_ok=True)
client = PersistentClient(path=str(DB_PATH))

if not chunks:
    print("⚠️ No chunks to index; loading existing vector store if present.")
    vectorstore = Chroma(client=client, collection_name=COLLECTION_NAME, embedding_function=embeddings)
else:
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass
    print("🆕 Building vector store from chunks...")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        client=client,
        collection_name=COLLECTION_NAME,
    )
print(f"✅ Vector store ready at {DB_PATH} with {vectorstore._collection.count()} embeddings")

🆕 Building vector store from chunks...
✅ Vector store ready at data/db with 102 embeddings


## Cell 6 — Retrieval & Answer Generation

In [33]:
def rerank(query: str, docs: list, top_k: int) -> list:
    from pydantic import BaseModel, Field
    class RankOrder(BaseModel):
        order: list[int] = Field(description="Ranked chunk IDs from 1 to N, most relevant first")
    user_prompt = f"Question:\n{query}\n\nChunks (rank by relevance to the question, most relevant first):\n"
    for i, doc in enumerate(docs):
        user_prompt += f"# CHUNK ID {i + 1}:\n{doc.page_content[:1500]}\n\n"
    user_prompt += "Reply with JSON only: {\"order\": [list of chunk IDs in relevance order]}."
    response = llm_client_rerank.chat.completions.create(
        model=LLM_RERANK_MODEL,
        messages=[{"role": "user", "content": user_prompt}],
        response_format={"type": "json_object"},
        temperature=0,
    )
    import json
    data = json.loads(response.choices[0].message.content)
    order = data.get("order", list(range(1, len(docs) + 1)))
    return [docs[i - 1] for i in order if 1 <= i <= len(docs)][:top_k]


def retrieve_and_answer(query: str, top_k: int | None = None) -> dict:
    top_k = top_k or RERANK_TOP_K
    docs = vectorstore.similarity_search(query, k=RETRIEVAL_K)
    if len(docs) > top_k:
        docs = rerank(query, docs, top_k)
    else:
        docs = docs[:top_k]
    context = "\n\n".join([doc.page_content for doc in docs])
    sources = [doc.metadata.get("url") or doc.metadata.get("source", "") for doc in docs]
    
    # Step 3: Generate answer with LLM
    system_prompt = """You are a helpful assistant answering questions based on provided documentation.
    Always cite your sources. If you don't know the answer, say so.
    
    Context from documentation:
{context}
"""
    
    messages = [
        {"role": "system", "content": system_prompt.format(context=context)},
        {"role": "user", "content": query}
    ]
    
    response = llm_client_chat.chat.completions.create(
        model=LLM_CHAT_MODEL,
        messages=messages,
        temperature=0.3,
        max_tokens=500,
    )
    answer = response.choices[0].message.content
    unique_sources = list(dict.fromkeys(sources))
    sources_text = "\n".join([f"- {s}" for s in unique_sources if s])
    return {
        "answer": answer,
        "sources": sources_text,
        "num_chunks_retrieved": len(docs),
        "docs": docs,
    }


def _retrieve_docs(query: str, top_k: int | None = None):
    top_k = top_k or RERANK_TOP_K
    docs = vectorstore.similarity_search(query, k=RETRIEVAL_K)
    if len(docs) > top_k:
        docs = rerank(query, docs, top_k)
    else:
        docs = docs[:top_k]
    return docs


def stream_rag_response(query: str):
    docs = _retrieve_docs(query)
    context = "\n\n".join([doc.page_content for doc in docs])
    system_prompt = """You are a helpful assistant answering questions based on provided documentation.
Always cite your sources. If you don't know the answer, say so.

Context from documentation:
{context}
"""
    messages = [
        {"role": "system", "content": system_prompt.format(context=context)},
        {"role": "user", "content": query},
    ]
    stream = llm_client_chat.chat.completions.create(
        model=LLM_CHAT_MODEL,
        messages=messages,
        temperature=0.3,
        max_tokens=500,
        stream=True,
    )
    acc = ""
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            acc += chunk.choices[0].delta.content
            yield acc, docs
    yield acc, docs


# Test the pipeline
test_query = "What are the important things to know about topfaith"
print(f"❓ Query: {test_query}\n")

result = retrieve_and_answer(test_query)
print(f"💬 Answer:\n{result['answer']}")
print(f"\n📚 Sources:\n{result['sources']}")

❓ Query: What are the important things to know about topfaith

 retrieved docs: [Document(id='173c1598-5856-4405-b07b-b0ce87fd8b89', metadata={'source': 'concept_and_objectives__topfaith_sc.md', 'category': 'school', 'url': 'https://topfaith.sch.ng/pg/about/concept-and-objectives'}, page_content='Additional Resources and Links\n\nTopfaith Schools provides various resources including history, philosophy, foundation programs, and reasons to choose the school. Contact details and social media links are also available for further engagement.\n\nFacebook\nQuick Links\nHistory And Philosophy\nTopfaith International University Foundation Programme\n25 Reasons To Choose TOPFAITH\nOur Anthem'), Document(id='3bd3f564-d164-4aaf-891d-598a7cee184a', metadata={'category': 'general', 'source': 'topfaith_legacy_college_tlc__topfai.md', 'url': 'https://topfaith.sch.ng/pg/about/topfaith-legacy-college'}, page_content="Additional Resources and Links\n\nThe college offers various resources including its h

## Cell 6b — Evaluation

Run retrieval and answer evals over `evaluation/tests.jsonl`. Requires vectorstore and `retrieve_and_answer` from above.

In [34]:
from evaluation.test import load_tests
from evaluation.eval import evaluate_retrieval, evaluate_answer, run_eval_summary

def fetch_context_for_eval(question: str, k: int = 10):
    return vectorstore.similarity_search(question, k=k)

def get_answer_for_eval(question: str) -> str:
    return retrieve_and_answer(question)["answer"]

summary = run_eval_summary(fetch_context_for_eval, get_answer_for_eval, k_retrieval=10)
print("Evaluation summary (RAG pipeline)")
print("Retrieval — avg MRR:", round(summary["retrieval"]["avg_mrr"], 4), "| avg keyword coverage %:", round(summary["retrieval"]["avg_keyword_coverage_pct"], 1))
print("Answer — avg accuracy:", round(summary["answer"]["avg_accuracy"], 2), "| completeness:", round(summary["answer"]["avg_completeness"], 2), "| relevance:", round(summary["answer"]["avg_relevance"], 2))
print("n_tests:", summary["n_tests"])

 retrieved docs: [Document(id='b997fa62-b9e1-4b92-a077-92f790aab8e6', metadata={'url': 'https://topfaith.sch.ng/', 'category': 'school', 'source': 'home__topfaith_schools.md'}, page_content="Admissions and Application Process\n\nApplication forms for entrance examinations into JS1 and transfers are available for a fee of ₦10,000, including past questions. Forms can be purchased at the school's office or other sales venues, with options to print or complete earlier applications online.\n\nApplication Forms are now on sale.\nApplication forms for entrance examinations into JS1 and Transfers are now on sale at a\n                    non-refundable fee of ₦ 10,000. The application form package includes: the application form\n                    and three(3) years JS1 entrance examination past questions.\na) Fill the application form\nApply\n                        Now!\nIf you want to print or complete an application made earlier,\nclick\n                        here\nOr\nVisit any of the 

## Cell 7 — Gradio UI

In [35]:
import pandas as pd
from evaluation.eval import evaluate_retrieval, evaluate_answer
from evaluation.test import load_tests

def format_sources_md(docs):
    if not docs:
        return "_Send a message in the Chat tab to see sources here._"
    lines = []
    for i, doc in enumerate(docs, 1):
        url = (doc.metadata.get("url") or "").strip()
        source = (doc.metadata.get("source") or "").strip()
        title = source or url or f"Source {i}"
        snippet = (doc.page_content or "")[:200].replace("\n", " ")
        if url:
            lines.append(f"{i}. **[{title}]({url})**\n   {snippet}...")
        else:
            lines.append(f"{i}. **{title}**\n   {snippet}...")
    return "\n\n".join(lines)


def chat_stream(history, message, last_docs):
    if not (message and message.strip()):
        return history, last_docs, format_sources_md(last_docs)
    history = history + [(message, None)]
    try:
        for partial, docs in stream_rag_response(message.strip()):
            last_docs = docs
            new_history = history[:-1] + [(message, partial)]
            yield new_history, last_docs, format_sources_md(last_docs)
    except Exception as e:
        new_history = history[:-1] + [(message, f"Error: {str(e)}")]
        yield new_history, last_docs, format_sources_md(last_docs)


def run_evaluation(progress=gr.Progress()):
    def fetch(question: str, k: int = 10):
        return vectorstore.similarity_search(question, k=k)

    def get_ans(question: str):
        return retrieve_and_answer(question)["answer"]

    tests = load_tests()
    n = len(tests)
    ret_mrrs, ret_cov = [], []
    ans_acc, ans_comp, ans_rel = [], [], []
    by_cat_mrr, by_cat_acc = {}, {}

    for i, test in enumerate(tests):
        progress((i + 1) / n, desc=f"Evaluating {i + 1}/{n}...")
        ret = evaluate_retrieval(test, fetch, k=10)
        ans_eval, _ = evaluate_answer(test, lambda q: get_ans(q))
        ret_mrrs.append(ret.mrr)
        ret_cov.append(ret.keyword_coverage)
        ans_acc.append(ans_eval.accuracy)
        ans_comp.append(ans_eval.completeness)
        ans_rel.append(ans_eval.relevance)
        by_cat_mrr.setdefault(test.category, []).append(ret.mrr)
        by_cat_acc.setdefault(test.category, []).append(ans_eval.accuracy)

    avg_mrr = sum(ret_mrrs) / n
    avg_cov = sum(ret_cov) / n
    avg_acc = sum(ans_acc) / n
    avg_comp = sum(ans_comp) / n
    avg_rel = sum(ans_rel) / n

    def color(v, green, amber):
        if v >= green:
            return "green"
        if v >= amber:
            return "orange"
        return "red"

    html = f"""
    <div style="padding: 10px;">
        <div style="margin: 10px 0; padding: 15px; background: #f5f5f5; border-radius: 8px; border-left: 5px solid {color(avg_mrr, 0.9, 0.75)};">
            <div style="font-size: 14px; color: #666;">Mean Reciprocal Rank (MRR)</div>
            <div style="font-size: 28px; font-weight: bold;">{avg_mrr:.4f}</div>
        </div>
        <div style="margin: 10px 0; padding: 15px; background: #f5f5f5; border-radius: 8px; border-left: 5px solid {color(avg_cov, 90, 75)};">
            <div style="font-size: 14px; color: #666;">Keyword Coverage</div>
            <div style="font-size: 28px; font-weight: bold;">{avg_cov:.1f}%</div>
        </div>
        <div style="margin: 10px 0; padding: 15px; background: #f5f5f5; border-radius: 8px; border-left: 5px solid {color(avg_acc, 4.5, 4)};">
            <div style="font-size: 14px; color: #666;">Answer Accuracy (1-5)</div>
            <div style="font-size: 28px; font-weight: bold;">{avg_acc:.2f}/5</div>
        </div>
        <div style="margin: 10px 0; padding: 15px; background: #f5f5f5; border-radius: 8px; border-left: 5px solid {color(avg_comp, 4.5, 4)};">
            <div style="font-size: 14px; color: #666;">Completeness</div>
            <div style="font-size: 28px; font-weight: bold;">{avg_comp:.2f}/5</div>
        </div>
        <div style="margin: 10px 0; padding: 15px; background: #f5f5f5; border-radius: 8px; border-left: 5px solid {color(avg_rel, 4.5, 4)};">
            <div style="font-size: 14px; color: #666;">Relevance</div>
            <div style="font-size: 28px; font-weight: bold;">{avg_rel:.2f}/5</div>
        </div>
        <p style="margin-top: 15px; color: #155724;"><b>Evaluation complete: {n} tests</b></p>
    </div>
    """
    df_mrr = pd.DataFrame([{"Category": c, "Average MRR": sum(s) / len(s)} for c, s in by_cat_mrr.items()])
    df_acc = pd.DataFrame([{"Category": c, "Average Accuracy": sum(s) / len(s)} for c, s in by_cat_acc.items()])
    return html, df_mrr, df_acc


with gr.Blocks(title="Smart Document RAG") as demo:
    gr.Markdown("# Smart Document RAG — Chat, Sources, Evaluation")

    last_retrieved_docs = gr.State(value=[])

    with gr.Tabs(selected=1):
        with gr.TabItem("Sources"):
            sources_out = gr.Markdown(
                value="_Send a message in the Chat tab to see sources here._",
                label="Sources used for the last response",
            )

        with gr.TabItem("Chat"):
            chatbot = gr.Chatbot(label="Chat", type="tuples")
            msg_in = gr.Textbox(placeholder="Ask about the documentation...", label="Message", lines=2)
            submit_btn = gr.Button("Send", variant="primary")
            submit_btn.click(
                chat_stream,
                inputs=[chatbot, msg_in, last_retrieved_docs],
                outputs=[chatbot, last_retrieved_docs, sources_out],
            )
            msg_in.submit(
                chat_stream,
                inputs=[chatbot, msg_in, last_retrieved_docs],
                outputs=[chatbot, last_retrieved_docs, sources_out],
            )

        with gr.TabItem("Evaluation"):
            eval_btn = gr.Button("Run Evaluation", variant="primary", size="lg")
            with gr.Row():
                eval_metrics = gr.HTML(value="<div style='padding: 20px; color: #999;'>Click Run Evaluation to start.</div>")
                with gr.Column():
                    eval_chart_mrr = gr.BarPlot(x="Category", y="Average MRR", title="MRR by Category", y_lim=[0, 1], height=300)
                    eval_chart_acc = gr.BarPlot(x="Category", y="Average Accuracy", title="Accuracy by Category", y_lim=[1, 5], height=300)
            eval_btn.click(
                run_evaluation,
                outputs=[eval_metrics, eval_chart_mrr, eval_chart_acc],
            )

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


 retrieved docs: [Document(id='1cd93174-6e3f-45b0-bb73-85e50ef5668a', metadata={'url': 'https://topfaith.sch.ng/pg/academics/remote-learning-details', 'category': 'access', 'source': 'remote_learning_details__topfaith_s.md'}, page_content='Remote Learning Overview and Contact Info\n\nThis section provides an overview of the remote learning setup at Topfaith Schools, including contact details and instructions for downloading Zoom. It introduces the remote learning platform and how to access classes via Zoom links and Google Classrooms.\n\n# Remote learning details | Topfaith Schools\n\n+234 706 6211 122\n+234 803 454 6317\ninfo@topfaith.sch.ng\nRemote Learning Details\nZoom can be dowloaded at\nhttps://zoom.us/download\nZoom Links\nClass\nZoom Link\nMeeting ID\nPassword\nJS1\nCLICK FOR JS1 REMOTE LEARNING\n845-3688-0450\nMathtiss\nJS2\nCLICK FOR JS2 REMOTE LEARNING\n870-9329-6409\nMathtiss\nJS3\nCLICK FOR JS3 REMOTE LEARNING\n897-1676-7490\nMathtiss\nSS1\nCLICK FOR SS1 REMOTE LEARNING\n

---

## Concepts Demonstrated

| Cell | Pipeline / API | What it shows |
|------|---------------|---------------|
| 3 | Crawl + HF Categorize | Same-domain crawl, stats, zero-shot category, data/source |
| 4 | Document Splitting | Load from data/source; RecursiveCharacterTextSplitter or LLM chunking (USE_LLM_CHUNKING) |
| 5 | Chroma DB | Vector storage in data/db |
| 6 | RAG Pipeline | Retrieve RETRIEVAL_K, LLM re-rank to RERANK_TOP_K, then generate |
| 6 | Citations | Source attribution |
| 6b | Evaluation | Retrieval (MRR, keyword coverage) + answer (LLM-as-judge) over evaluation/tests.jsonl |
| 7 | Gradio UI | Chat tab (streaming, history), Sources tab (extracted docs + links), Evaluation tab (Run + charts); configurable LLM per task (chunking, rerank, chat) |

---

## Extensions Possible

1. **Hybrid search**: Combine semantic + keyword search
2. **Query rewriting**: Expand/rewrite query before retrieval
3. **Multi-modal**: Support PDFs, images
4. **Custom embeddings**: Use local embedding models
